<a href="https://colab.research.google.com/github/rfcastrovera/BIGDATA/blob/main/Lab5_Recomendadores_ALS_Embeddings_MovieLens_rcastrov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5: Recomendadores con ALS y búsqueda semántica con embeddings

**Análisis de Big Data · Magíster en Data Science UDD · Sesión 5 (viernes 11 de septiembre)**

Alumno: Ricardo Castro Vera

Este laboratorio cambia de dataset por primera vez en el curso: **MovieLens pequeño**
(`ml-latest-small`: 610 usuarios, 9.742 películas, 100.836 calificaciones). La razón es de método,
no de comodidad: el objetivo de aprendizaje es el **criterio**: qué cambia al mover `rank`, qué
pasa con un usuario nuevo, cuándo el contenido complementa a la interacción. El criterio se
aprende iterando. Con 100 mil calificaciones cada ajuste toma segundos; en 20 minutos caben tres
configuraciones comparadas.

Las dos mitades trabajan sobre **el mismo catálogo**: **(a)** filtrado colaborativo
(*collaborative filtering*) con ALS y tracking en MLflow; **(b)** representaciones vectoriales
(*embeddings*) de título y género con `sentence-transformers` y búsqueda semántica. La misma
película recomendada por interacción y por contenido, y el usuario sin historial resuelto por la
segunda vía.

---

## Qué se entrega el miércoles 16 de septiembre, 23:59

**No se entrega el notebook completo.** Se entrega:

1. La **tabla de `mlflow.search_runs()`** con **al menos tres configuraciones de ALS** y su RMSE
   (sección 10).
2. El **top-5 de una consulta semántica propia** (sección 9, celda `MI_CONSULTA`).
3. **Un párrafo** que relacione ambas: qué le recomendaría ALS a un usuario nuevo y por qué la
   búsqueda por contenido cubre ese caso (sección 10b, celda de texto a completar).

El **Quiz 2 es el lunes 14**: el fin de
semana es para el quiz; el laboratorio vence después, a propósito.

---

## Objetivos

- Entrenar un recomendador por factorización de matrices con `ALS` y entender qué se distribuye
  entre ejecutores y por qué el algoritmo alterna.
- Reconocer el **arranque en frío** (*cold start*) en su forma más concreta, un `NaN` en el RMSE,
  y la decisión que lo resuelve.
- Medir el **sesgo de popularidad** con la cobertura del catálogo, no con la intuición.
- Calcular embeddings **a escala** con una función definida por el usuario de pandas
  (*pandas UDF*) de tipo iterador: el modelo se carga una vez por partición, no una vez por fila.
- Comparar las tres configuraciones de ALS **en la misma tabla** de MLflow, con métrica y tiempo:
  nuevamente aparece `search_runs()`, que será una exigencia en la Fase 2.

## 0. Setup

Misma receta del Lab 4 más una instalación nueva: `sentence-transformers`, que trae el modelo de
embeddings. Demora un par de minutos.

Los *runs* se leen con `mlflow.search_runs()`.

In [2]:
# Colab. En una instalación local con PySpark ya presente, omitir el bloque de instalación.
import sys
print("Python", ".".join(map(str, sys.version_info[:3])))

# pyarrow solo como wheel; PySpark y MLflow fijados: la misma combinación que funcionó en el Lab 4.
!pip -q install --only-binary=:all: pyarrow
!pip -q install --prefer-binary pyspark==3.5.1 "mlflow>=2.16"
# Nuevo en este lab: el modelo de embeddings (usa el torch que Colab ya trae).
!pip -q install sentence-transformers

import time, os
import numpy as np, pandas as pd
import mlflow, mlflow.spark
from pyspark.sql import SparkSession, functions as F, types as T

spark = (SparkSession.builder
         .appName("lab5-recomendadores")
         .config("spark.sql.shuffle.partitions", "16")      # dataset chico: menos particiones, menos overhead
         .config("spark.driver.memory", "6g")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| MLflow", mlflow.__version__)

Python 3.13.15
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 19.1 MB

In [3]:
# La misma celda de tracking del Lab 4. Solo cambia el nombre del experimento.
RUTA_DB = "/content/mlflow.db"

mlflow.set_tracking_uri(f"sqlite:///{RUTA_DB}")
mlflow.set_experiment("lab5_recomendadores")
print("Tracking URI:", mlflow.get_tracking_uri())

# Alternativa si SQLite diera problemas: store de archivos, que exige el opt-out.
# import os; os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
# mlflow.set_tracking_uri("file:/content/mlruns")

# Respaldo opcional al terminar: conserva los runs si se reinicia el entorno.
# from google.colab import drive; drive.mount("/content/drive")
# !cp {RUTA_DB} /content/drive/MyDrive/adbd/mlflow_lab5.db

2026/09/25 22:08:59 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/25 22:08:59 INFO mlflow.store.db.utils: Updating database tables
2026/09/25 22:09:04 INFO mlflow.tracking.fluent: Experiment with name 'lab5_recomendadores' does not exist. Creating a new experiment.


Tracking URI: sqlite:////content/mlflow.db


## 1. Los datos: una matriz casi vacía

MovieLens pequeño se descarga directo de GroupLens (1 MB). Dos archivos importan:
`ratings.csv` (usuario, película, nota de 0,5 a 5, marca de tiempo) y `movies.csv` (título y géneros).

La cifra que ordena toda la primera mitad es la **densidad** de la matriz usuario–ítem: qué
fracción de las celdas tiene una calificación. El recomendador existe para llenar las demás.

In [4]:
import os, ssl, shutil, zipfile, urllib.request

URL_ML   = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
RUTA_ZIP = "/content/ml-latest-small.zip"
RUTA_ML  = "/content/ml-latest-small"
ENLACE_RESPALDO = ""   # si nada de lo anterior funciona: enlace de Drive al mismo zip (receta gdown del Lab 4)

def zip_valido(ruta):
    """Un zip servido a medias o una página de error pesan poco y no abren como zip."""
    return os.path.exists(ruta) and os.path.getsize(ruta) > 100_000 and zipfile.is_zipfile(ruta)

def descargar(url, destino, verificar_ssl=True):
    contexto = None if verificar_ssl else ssl._create_unverified_context()
    pedido = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(pedido, timeout=90, context=contexto) as r, open(destino, "wb") as f:
        shutil.copyfileobj(r, f)

if not os.path.exists(f"{RUTA_ML}/ratings.csv"):
    # El certificado de GroupLens caduca seguido: si la verificación falla, se reintenta sin ella.
    for etiqueta, kwargs in [("con verificación", {}), ("sin verificación de certificado", {"verificar_ssl": False})]:
        if zip_valido(RUTA_ZIP):
            break
        if os.path.exists(RUTA_ZIP):
            os.remove(RUTA_ZIP)
        try:
            descargar(URL_ML, RUTA_ZIP, **kwargs)
            print(f"descargado de GroupLens ({etiqueta}): {os.path.getsize(RUTA_ZIP)/1e6:.1f} MB")
        except Exception as e:
            print(f"intento {etiqueta}:", e)
    if not zip_valido(RUTA_ZIP) and ENLACE_RESPALDO:
        !gdown --fuzzy "$ENLACE_RESPALDO" -O "$RUTA_ZIP"
    if not zip_valido(RUTA_ZIP):
        raise RuntimeError("El zip no llegó completo. Pegar en ENLACE_RESPALDO el enlace de Drive al mismo ml-latest-small.zip y volver a ejecutar esta celda.")
    with zipfile.ZipFile(RUTA_ZIP) as z:
        z.extractall("/content")

print(sorted(os.listdir(RUTA_ML)))

descargado de GroupLens (con verificación): 1.0 MB
['README.txt', 'links.csv', 'movies.csv', 'ratings.csv', 'tags.csv']


In [5]:
ratings = (spark.read.csv(f"{RUTA_ML}/ratings.csv", header=True, inferSchema=True)
           .select("userId", "movieId", "rating", "timestamp"))
movies  = spark.read.csv(f"{RUTA_ML}/movies.csv", header=True, inferSchema=True)

n_r = ratings.count()
n_u = ratings.select("userId").distinct().count()
n_m = movies.count()
print(f"calificaciones: {n_r:,} | usuarios: {n_u:,} | películas en catálogo: {n_m:,}")
print(f"densidad de la matriz usuario-ítem: {n_r / (n_u * n_m):.2%}  →  el {1 - n_r / (n_u * n_m):.1%} está vacío")

ratings.show(5)
movies.show(5, truncate=60)

calificaciones: 100,836 | usuarios: 610 | películas en catálogo: 9,742
densidad de la matriz usuario-ítem: 1.70%  →  el 98.3% está vacío
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
+------+-------+------+---------+
only showing top 5 rows

+-------+----------------------------------+-------------------------------------------+
|movieId|                             title|                                     genres|
+-------+----------------------------------+-------------------------------------------+
|      1|                  Toy Story (1995)|Adventure|Animation|Children|Comedy|Fantasy|
|      2|                    Jumanji (1995)|                 Adventure|Children|Fantasy|
|      3|           Grumpier Old Men (1995)|                             Comedy|Romance

## 2. El split, y una advertencia sobre el tiempo

Se usa `randomSplit` 80/20 con semilla fija, que es el protocolo estándar para comparar RMSE
entre configuraciones de ALS y el que permite que las tablas de distintos estudiantes sean
comparables.

Conviene decir lo que ese protocolo esconde. Las calificaciones tienen marca de tiempo, y en
producción el recomendador predice **lo que el usuario calificará después** de lo que ya calificó.
El split honesto para ese escenario es temporal por usuario: entrenar con sus calificaciones
antiguas, evaluar con las recientes, exactamente la lógica del corte del Lab 3.

In [6]:
train, test = ratings.randomSplit([0.8, 0.2], seed=42)
train.cache(); test.cache()
print(f"train: {train.count():,}   test: {test.count():,}")

# Películas que están en test y no en train: ALS no tiene cómo predecirlas. Este número
# explica el NaN de la sección siguiente.
items_train = train.select("movieId").distinct()
solo_test = test.select("movieId").distinct().join(items_train, "movieId", "left_anti").count()
print("películas presentes solo en test:", solo_test)

train: 80,578   test: 20,258
películas presentes solo en test: 770


## 3. ALS y el primer error que se ve

**Qué hace ALS.** Busca dos matrices de rango `k` —una fila por usuario, una fila por película—
cuyo producto aproxime la matriz de calificaciones en las celdas conocidas. El problema conjunto
no es convexo, pero **fijada una de las matrices, la otra se obtiene con mínimos cuadrados
ordinarios, fila por fila y de forma independiente**. Por eso el algoritmo alterna (*Alternating
Least Squares*): fija usuarios y resuelve películas, fija películas y resuelve usuarios. Esas
regresiones independientes son lo que se reparte entre ejecutores, y es la razón de que MLlib
traiga ALS y no descenso de gradiente para este problema.

**Los tres hiperparámetros que importan:** `rank` (la dimensión `k`), `regParam` (regularización)
y `maxIter` (alternancias). Y una decisión que no es hiperparámetro: `coldStartStrategy`.

Por defecto vale `"nan"`: cuando un usuario o una película del test no estaba en el entrenamiento,
ALS no tiene un vector para ella y predice `NaN`. El evaluador promedia y **el RMSE completo sale
NaN**. Es el primer error del lab, y se ve a propósito.

In [7]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

rmse_eval = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")

# Sin coldStartStrategy: el valor por defecto es "nan".
als_nan  = ALS(userCol="userId", itemCol="movieId", ratingCol="rating",
               rank=10, regParam=0.1, maxIter=10, seed=42)
pred_nan = als_nan.fit(train).transform(test)
print("RMSE con coldStartStrategy='nan':", rmse_eval.evaluate(pred_nan))
print("filas con predicción NaN:", pred_nan.filter(F.isnan("prediction")).count())
pred_nan.filter(F.isnan("prediction")).show(3)

RMSE con coldStartStrategy='nan': nan
filas con predicción NaN: 821
+------+-------+------+----------+----------+
|userId|movieId|rating| timestamp|prediction|
+------+-------+------+----------+----------+
|    18|  96488|   4.0|1515532582|       NaN|
|    18| 157110|   4.5|1485100549|       NaN|
|   186|   3711|   4.0|1031072998|       NaN|
+------+-------+------+----------+----------+
only showing top 3 rows



In [8]:
# coldStartStrategy="drop": las filas sin predicción posible se excluyen de la evaluación.
als_base = ALS(userCol="userId", itemCol="movieId", ratingCol="rating",
               rank=10, regParam=0.1, maxIter=10, seed=42, coldStartStrategy="drop")

t0 = time.time()
modelo_base = als_base.fit(train)
seg = time.time() - t0

pred = modelo_base.transform(test)
print(f"RMSE con 'drop': {rmse_eval.evaluate(pred):.4f}   entrenamiento: {seg:.1f} s")
print(f"filas evaluadas: {pred.count():,} de {test.count():,} en test")

# Lo que ALS aprendió: un vector de 'rank' números por usuario y por película.
print("factores por usuario:", modelo_base.userFactors.count(), "| por película:", modelo_base.itemFactors.count())
modelo_base.itemFactors.show(2, truncate=90)

RMSE con 'drop': 0.8814   entrenamiento: 6.9 s
filas evaluadas: 19,437 de 20,258 en test
factores por usuario: 610 | por película: 8954
+---+------------------------------------------------------------------------------------------+
| id|                                                                                  features|
+---+------------------------------------------------------------------------------------------+
| 10|[-0.4890948, -0.2404774, 0.1702258, -0.38345185, -0.17664507, -0.43825656, -0.9671927, ...|
| 20|[-1.0034562, -0.3965999, -0.3616333, -0.0116487, -0.16513397, -0.22424151, -0.39250734,...|
+---+------------------------------------------------------------------------------------------+
only showing top 2 rows



## 4. La función `experimento`, otra vez

La misma estructura del Lab 4 —ajustar, evaluar, cronometrar, registrar— con dos métricas propias
de un recomendador:

- **`rmse_test`**: la métrica del laboratorio. Mide qué tan bien se predice la nota.
- **`cobertura_top10`**: qué fracción del catálogo aparece al menos una vez en el top-10 de algún
  usuario. Es la medida más simple del **sesgo de popularidad**: un recomendador que le muestra a
  todos las mismas 200 películas tiene cobertura del 2% aunque su RMSE sea excelente. La métrica
  del negocio es un *ranking*, no una nota, y esta columna lo recuerda.

Y la tercera, que ya no es nueva: **`seg_entrenamiento`**, junto a las otras dos, para que la
comparación de la sección 10 pueda considerar el costo.

In [9]:
N_CATALOGO = n_m

def cobertura_catalogo(modelo, k=10):
    """Fracción del catálogo que aparece al menos una vez en el top-k de algún usuario."""
    recs = modelo.recommendForAllUsers(k)
    distintos = recs.select(F.explode("recommendations.movieId").alias("movieId")).distinct().count()
    return distintos / N_CATALOGO

def experimento(nombre, rank, regParam, maxIter=10, datos_train=None, datos_test=None):
    """Ajusta ALS, lo evalúa y registra todo en un run de MLflow. Devuelve (ALSModel, dict)."""
    dtr = datos_train if datos_train is not None else train
    dte = datos_test  if datos_test  is not None else test

    with mlflow.start_run(run_name=nombre):
        als = ALS(userCol="userId", itemCol="movieId", ratingCol="rating",
                  rank=rank, regParam=regParam, maxIter=maxIter,
                  coldStartStrategy="drop", seed=42)

        t0 = time.time()
        modelo = als.fit(dtr)
        seg_fit = time.time() - t0

        rmse = rmse_eval.evaluate(modelo.transform(dte))
        cob  = cobertura_catalogo(modelo)

        # --- Parámetros: las decisiones ---
        mlflow.log_param("tipo", "als")
        mlflow.log_param("dataset", "ml-latest-small")
        mlflow.log_param("rank", rank)
        mlflow.log_param("regParam", regParam)
        mlflow.log_param("maxIter", maxIter)
        mlflow.log_param("coldStartStrategy", "drop")

        # --- Métricas: predictiva, de negocio y computacional ---
        mlflow.log_metric("rmse_test", rmse)
        mlflow.log_metric("cobertura_top10", cob)
        mlflow.log_metric("seg_entrenamiento", seg_fit)
        mlflow.log_metric("filas_train", dtr.count())

        # --- Artefacto: el modelo exacto, recuperable ---
        mlflow.spark.log_model(modelo, "als")

        print(f"{nombre:18s} rank={rank:<3d} reg={regParam:<5}  RMSE={rmse:.4f}  cobertura={cob:.1%}  fit={seg_fit:5.1f}s")
        return modelo, {"rmse": rmse, "cobertura": cob, "seg": seg_fit}

## 5. Tres configuraciones deliberadamente distintas

| Run | Qué cambia | Qué pregunta responde |
|---|---|---|
| `als_base` | `rank=10`, `regParam=0.1` | ¿Cuánto explica un modelo pequeño? |
| `als_rank_alto` | `rank=50`, misma regularización | ¿Más dimensiones mejoran el RMSE, o solo cuestan tiempo y sobreajustan? |
| `als_reg_alta` | `rank=10`, `regParam=0.5` | ¿Qué le hace la regularización al RMSE **y a la cobertura**? |

El split, la semilla y `maxIter` se mantienen fijos: es lo que permite atribuir la diferencia a la
configuración. Cada corrida toma segundos, así que hay tiempo para agregar una cuarta propia.

In [10]:
modelo_A, m_A = experimento("als_base",       rank=10, regParam=0.1)

als_base           rank=10  reg=0.1    RMSE=0.8814  cobertura=6.8%  fit=  7.3s


In [11]:
modelo_B, m_B = experimento("als_rank_alto",  rank=50, regParam=0.1)

als_rank_alto      rank=50  reg=0.1    RMSE=0.8736  cobertura=13.0%  fit=  8.0s


In [12]:
modelo_C, m_C = experimento("als_reg_alta",   rank=10, regParam=0.5)

als_reg_alta       rank=10  reg=0.5    RMSE=0.9978  cobertura=0.3%  fit=  4.7s


In [13]:
modelo_D, m_D = experimento("als_propio", rank=20, regParam=0.05)

als_propio         rank=20  reg=0.05   RMSE=0.9486  cobertura=13.6%  fit=  4.8s


In [14]:
# Cuarta configuración, propia (opcional pero recomendable): cambiar UNA cosa y anotar por qué.
# modelo_D, m_D = experimento("als_propio", rank=20, regParam=0.05)

## 6. Lo que se recomienda, y a quién se le recomienda lo mismo

El RMSE es la métrica del laboratorio; la del negocio es un *ranking*: `recommendForAllUsers(k)`
devuelve el top-k de cada usuario. Dos lecturas sobre esa salida:

1. **Para un usuario concreto**: qué calificó alto y qué se le recomienda. La coherencia se ve a ojo.
2. **Para todos a la vez**: qué películas se repiten en los top-10. Si las más recomendadas son
   también las más calificadas, hay sesgo de popularidad; si son películas con dos o tres
   calificaciones, el problema es el inverso —recomendaciones sostenidas en poca evidencia—.
   Ambas son señales de monitoreo, y ambas se leen en la misma tabla.

In [15]:
modelo = modelo_A
recs = modelo.recommendForAllUsers(10).cache()

USUARIO = 1
print(f"Lo que el usuario {USUARIO} calificó con 4,5 o más:")
(train.filter((F.col("userId") == USUARIO) & (F.col("rating") >= 4.5))
      .join(movies, "movieId").select("title", "genres", "rating").show(6, truncate=48))

print(f"Lo que ALS le recomienda:")
(recs.filter(F.col("userId") == USUARIO)
     .select(F.explode("recommendations").alias("r"))
     .select(F.col("r.movieId").alias("movieId"), F.round(F.col("r.rating"), 2).alias("prediccion"))
     .join(movies, "movieId").orderBy(F.desc("prediccion"))
     .select("title", "genres", "prediccion").show(10, truncate=48))

Lo que el usuario 1 calificó con 4,5 o más:
+-----------------------------------------+-----------------------+------+
|                                    title|                 genres|rating|
+-----------------------------------------+-----------------------+------+
|              Seven (a.k.a. Se7en) (1995)|       Mystery|Thriller|   5.0|
|               Usual Suspects, The (1995)| Crime|Mystery|Thriller|   5.0|
|                    Canadian Bacon (1995)|             Comedy|War|   5.0|
|                         Desperado (1995)| Action|Romance|Western|   5.0|
|                     Billy Madison (1995)|                 Comedy|   5.0|
|Star Wars: Episode IV - A New Hope (1977)|Action|Adventure|Sci-Fi|   5.0|
+-----------------------------------------+-----------------------+------+
only showing top 6 rows

Lo que ALS le recomienda:
+------------------------------------------------+---------------------------------------+----------+
|                                           title|   

In [16]:
# Sesgo de popularidad: qué películas se repiten en los top-10, y cuánta evidencia las sostiene.
frecuencia  = (recs.select(F.explode("recommendations.movieId").alias("movieId"))
                   .groupBy("movieId").count().withColumnRenamed("count", "veces_recomendada"))
popularidad = train.groupBy("movieId").count().withColumnRenamed("count", "n_calificaciones")

(frecuencia.join(popularidad, "movieId", "left").join(movies, "movieId")
           .orderBy(F.desc("veces_recomendada"))
           .select("title", "veces_recomendada", "n_calificaciones").show(10, truncate=48))

print(f"cobertura del catálogo en el top-10 (als_base): {m_A['cobertura']:.1%}")
print(f"cobertura del catálogo en el top-10 (als_reg_alta): {m_C['cobertura']:.1%}")

+------------------------------------------------+-----------------+----------------+
|                                           title|veces_recomendada|n_calificaciones|
+------------------------------------------------+-----------------+----------------+
|Dragon Ball Z: The History of Trunks (Doragon...|              255|               1|
|                             On the Beach (1959)|              240|               1|
|Three Billboards Outside Ebbing, Missouri (2017)|              197|               6|
|                              Saving Face (2004)|              156|               1|
|                                          Cosmos|              140|               2|
|                         De platte jungle (1978)|              125|               1|
|                              Bitter Lake (2015)|              121|               1|
|   Andalusian Dog, An (Chien andalou, Un) (1929)|              121|               2|
|                                Watermark (2014)|    

### 6b. El usuario nuevo: la pregunta que ALS no puede responder

Un usuario que llega hoy no tiene fila en la matriz, así que no tiene vector. Con
`coldStartStrategy="drop"` la respuesta no es un error: es **cero filas**. Es la forma honesta del
arranque en frío, y la segunda mitad del laboratorio existe para resolverla.

In [17]:
NUEVO = 999_999    # un id que no existe en el entrenamiento
print("filas en recommendForAllUsers para el usuario nuevo:", recs.filter(F.col("userId") == NUEVO).count())

pedido = spark.createDataFrame([(NUEVO, 1), (NUEVO, 318)], ["userId", "movieId"])
print("filas que devuelve transform() para él:", modelo.transform(pedido).count(), "→ sin historial no hay factores, y sin factores no hay predicción")

filas en recommendForAllUsers para el usuario nuevo: 0
filas que devuelve transform() para él: 0 → sin historial no hay factores, y sin factores no hay predicción


## 7. Embeddings a escala: el modelo se carga una vez por partición

Aquí termina el aprendizaje sobre tablas y empieza el análisis de datos no estructurados. Un
*embedding* es un vector denso —384 números con `all-MiniLM-L6-v2`— que captura significado, de
modo que textos parecidos quedan cerca. Cómo se entrena ese modelo no es contenido del curso;
**cómo se aplica a millones de textos sí lo es**.

La herramienta es la **pandas UDF de tipo iterador**: recibe un iterador de lotes (`Iterator[pd.Series]`)
y devuelve otro. La diferencia con una UDF fila a fila está en la primera línea del cuerpo: el
modelo se carga **una vez por partición** y atiende todos los lotes que le llegan. Sobre 9.742
títulos la diferencia es de segundos; sobre 50 millones de descripciones de producto es la
diferencia entre 20 minutos y 20 horas.

El texto a codificar es título más géneros. Es poco —ya se verá qué alcanza a capturar—, y es
exactamente el tipo de atributo que un catálogo real siempre tiene, incluso para el producto que
se lanzó ayer y nadie ha comprado.

In [18]:
from typing import Iterator
from pyspark.sql.functions import pandas_udf

MODELO_ST = "all-MiniLM-L6-v2"

catalogo = movies.withColumn(
    "texto", F.concat_ws(" · ", "title", F.regexp_replace("genres", r"\|", ", ")))
catalogo.select("texto").show(3, truncate=80)

def embeddings_por_particion(df):
    """El modelo se carga una vez por partición, igual que en una pandas UDF de iterador."""
    esquema = T.StructType(list(df.schema.fields) +
                           [T.StructField("embedding", T.ArrayType(T.FloatType()))])
    def por_particion(filas):
        from sentence_transformers import SentenceTransformer
        modelo_st = SentenceTransformer(MODELO_ST)   # una vez por partición
        filas = list(filas)
        if not filas:
            return []
        vecs = modelo_st.encode([f["texto"] for f in filas], batch_size=64, normalize_embeddings=True)
        return [tuple(f) + ([float(x) for x in v],) for f, v in zip(filas, vecs)]
    return spark.createDataFrame(df.rdd.mapPartitions(por_particion), esquema)

+------------------------------------------------------------------+
|                                                             texto|
+------------------------------------------------------------------+
|Toy Story (1995) · Adventure, Animation, Children, Comedy, Fantasy|
|                     Jumanji (1995) · Adventure, Children, Fantasy|
|                         Grumpier Old Men (1995) · Comedy, Romance|
+------------------------------------------------------------------+
only showing top 3 rows



In [19]:
# repartition(2): dos particiones, dos cargas del modelo, dos núcleos en Colab. Con 200 particiones
# en un cluster serían 200 cargas — sigue siendo 200, y no 50 millones.
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

t0 = time.time()
emb = embeddings_por_particion(catalogo.repartition(2)).cache()
n_emb = emb.count()
seg_emb = time.time() - t0
print(f"{n_emb:,} embeddings en {seg_emb:.0f} s (incluye cargar el modelo en cada partición)")

emb.select("title", F.size("embedding").alias("dim"),
           F.slice("embedding", 1, 4).alias("primeros_4")).show(3, truncate=60)

# El costo también se registra: la tabla de la sección 10 lo muestra junto a los de ALS.
with mlflow.start_run(run_name="embeddings_minilm"):
    mlflow.log_param("tipo", "contenido")
    mlflow.log_param("modelo", MODELO_ST)
    mlflow.log_param("dim", 384)
    mlflow.log_metric("seg_embeddings", seg_emb)
    mlflow.log_metric("n_items", n_emb)

9,742 embeddings en 107 s (incluye cargar el modelo en cada partición)
+----------------------------------+---+------------------------------------------------------+
|                             title|dim|                                            primeros_4|
+----------------------------------+---+------------------------------------------------------+
|     Holiday (Jour de fête) (1949)|384| [0.024446795, 0.094775744, -1.6442515E-4, 0.06380555]|
|Exorcism of Emily Rose, The (2005)|384|[-0.001845138, -0.051417757, -0.046493255, 0.05621807]|
|                    Cabaret (1972)|384|   [0.06837621, 0.038821913, -0.05819256, 0.010271236]|
+----------------------------------+---+------------------------------------------------------+
only showing top 3 rows



## 8. Búsqueda semántica: similitud coseno sobre el catálogo

Los vectores se normalizaron al codificar, así que la **similitud coseno es un producto punto**.
Con 9.742 × 384 números la matriz completa cabe en memoria del driver y una consulta es una
multiplicación de matrices; a escala real esa multiplicación se reemplaza por un **índice
vectorial** (FAISS en local, o una base vectorial como las vistas en Big Data y Cloud Computing).
El puente queda como extensión E3; la lógica es la misma.

In [20]:
pdf = emb.select("movieId", "title", "genres", "embedding").toPandas()
MATRIZ  = np.vstack(pdf["embedding"].to_list()).astype(np.float32)     # 9.742 × 384
IDS     = pdf["movieId"].to_numpy()
TITULOS = pdf["title"].to_numpy()
GENEROS = pdf["genres"].to_numpy()
print(MATRIZ.shape, "· normas ≈ 1:", np.round(np.linalg.norm(MATRIZ[:3], axis=1), 3))

from sentence_transformers import SentenceTransformer
st_driver = SentenceTransformer(MODELO_ST)      # una copia en el driver, solo para codificar consultas

def buscar(consulta, k=5):
    """Top-k del catálogo más cercano a una frase libre."""
    q = st_driver.encode([consulta], normalize_embeddings=True)[0]
    sims = MATRIZ @ q
    idx = np.argsort(-sims)[:k]
    return pd.DataFrame({"movieId": IDS[idx], "titulo": TITULOS[idx],
                         "generos": GENEROS[idx], "similitud": sims[idx].round(3)})

def similares(movie_id, k=5):
    """Top-k de películas vecinas a una dada, por contenido."""
    i = int(np.where(IDS == movie_id)[0][0])
    sims = MATRIZ @ MATRIZ[i]
    idx = np.argsort(-sims)[1:k + 1]          # excluye la propia película
    return pd.DataFrame({"titulo": TITULOS[idx], "generos": GENEROS[idx], "similitud": sims[idx].round(3)})

buscar("space adventure with robots and a rebellion")

(9742, 384) · normas ≈ 1: [1. 1. 1.]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,movieId,titulo,generos,similitud
0,136800,Robot Overlords (2014),Action|Adventure|Sci-Fi,0.668
1,32031,Robots (2005),Adventure|Animation|Children|Comedy|Fantasy|Sc...,0.610
2,8644,"I, Robot (2004)",Action|Adventure|Sci-Fi|Thriller,0.579
3,8426,Robot Carnival (Roboto kânibauru) (1987),Animation|Comedy|Drama|Fantasy|Sci-Fi,0.548
4,122900,Ant-Man (2015),Action|Adventure|Sci-Fi,0.525


In [21]:
# La favorita del usuario 1 según ALS (sección 6) y sus vecinas por contenido: las dos vías
# sobre las mismas películas. Comparar esta lista con la que ALS le recomendó.
fav = (train.filter(F.col("userId") == USUARIO)
            .orderBy(F.desc("rating"), F.desc("timestamp")).first()).movieId
print("favorita del usuario", USUARIO, "→", TITULOS[IDS == fav][0])
similares(fav)

favorita del usuario 1 → Tombstone (1993)


,titulo,generos,similitud
0,"Walk Among the Tombstones, A (2014)",Action|Crime|Mystery|Thriller,0.665
1,Shallow Grave (1994),Comedy|Drama|Thriller,0.618
2,Last Action Hero (1993),Action|Adventure|Comedy|Fantasy,0.608
3,Alive (1993),Drama,0.606
4,Wyatt Earp (1994),Western,0.606


## 9. El usuario nuevo, resuelto por contenido

El mismo usuario para el que ALS devolvió cero filas. Basta **una frase suya** —lo que busca, o el
título de una película que le gustó— para tener un top-5 razonable, porque el contenido del
catálogo existe desde el día uno aunque nadie lo haya calificado.

Esto es **recomendación por contenido** (*content-based*), y en producción convive con ALS: la
primera cubre el arranque en frío y el surtido nuevo; la segunda aprende lo que el contenido no
dice —qué películas se ven juntas aunque no se parezcan—. El sistema resultante se llama
**híbrido**, y ese es el criterio de decisión de la sesión:

- **ALS** cuando abunda la interacción.
- **Embeddings** cuando hay texto o atributos y poca interacción.
- **Híbrido** cuando hay ambos, que es casi siempre.

Una nota honesta sobre el modelo: `all-MiniLM-L6-v2` fue entrenado sobre texto en inglés y el
catálogo está en inglés, así que las consultas en inglés funcionan mejor. La variante multilingüe
queda en la extensión E4.

In [22]:
buscar("dark psychological thriller with an unreliable narrator")

,movieId,titulo,generos,similitud
0,149334,Nocturnal Animals,Drama|Thriller,0.603
1,62299,Alone in the Dark II (2008),Action|Horror,0.593
2,74545,"Ghost Writer, The (2010)",Drama|Mystery|Thriller,0.592
3,118082,The Voices (2014),Comedy|Crime|Thriller,0.586
4,6966,Darkman (1990),Action|Crime|Fantasy|Sci-Fi|Thriller,0.582


In [26]:
# ENTREGA · pieza 2: escribir aquí una consulta propia. La tabla que devuelve es parte de la entrega.
MI_CONSULTA = "detectives solving mysterious crimes in Victorian London"
buscar(MI_CONSULTA)

,movieId,titulo,generos,similitud
0,185135,Sherlock - A Study in Pink (2010),Crime,0.573
1,147300,Adventures Of Sherlock Holmes And Dr. Watson: ...,Crime|Mystery,0.545
2,147328,The Adventures of Sherlock Holmes and Dr. Wats...,Crime,0.544
3,60737,Watching the Detectives (2007),Comedy|Romance,0.539
4,172875,A Detective Story (2003),Animation|Sci-Fi,0.527


## 10. `search_runs()`: la tabla comparativa — **este es el entregable**

Tercera vez que se lee un experimento como consulta y no como recuerdo. La tabla muestra las
configuraciones de ALS con sus tres métricas —RMSE, cobertura y tiempo— y, en la última fila, el
costo de calcular los embeddings. La comparación que se pide no es "cuál tiene el RMSE más bajo",
sino **cuál conviene, y para qué usuario**.

In [24]:
pd.set_option("display.width", 180)

runs = mlflow.search_runs(order_by=["metrics.rmse_test ASC"])

COLS = ["tags.mlflow.runName", "params.tipo", "params.rank", "params.regParam", "params.maxIter",
        "metrics.rmse_test", "metrics.cobertura_top10", "metrics.seg_entrenamiento",
        "params.modelo", "metrics.seg_embeddings"]
tabla = runs[[c for c in COLS if c in runs.columns]].copy()
tabla.columns = [c.split(".")[-1] for c in tabla.columns]
tabla = tabla.rename(columns={"runName": "experimento"}).round(4)

print(f"runs registrados: {len(runs)}")
tabla

runs registrados: 5


,experimento,tipo,rank,regParam,maxIter,rmse_test,cobertura_top10,seg_entrenamiento,modelo,seg_embeddings
0,als_rank_alto,als,50,0.1,10,0.8736,0.1295,8.0457,None,NaN
1,als_base,als,10,0.1,10,0.8814,0.0683,7.3459,None,NaN
2,als_propio,als,20,0.05,10,0.9486,0.1359,4.7800,None,NaN
3,als_reg_alta,als,10,0.5,10,0.9978,0.0032,4.6565,None,NaN
4,embeddings_minilm,contenido,None,None,None,NaN,NaN,NaN,all-MiniLM-L6-v2,107.3754


In [25]:
# Exportar la tabla: es el archivo que se adjunta en Canvas.
tabla.to_csv("lab5_comparacion_als.csv", index=False)
print(tabla.to_markdown(index=False))

| experimento       | tipo      |   rank |   regParam |   maxIter |   rmse_test |   cobertura_top10 |   seg_entrenamiento | modelo           |   seg_embeddings |
|:------------------|:----------|-------:|-----------:|----------:|------------:|------------------:|--------------------:|:-----------------|-----------------:|
| als_rank_alto     | als       |     50 |       0.1  |        10 |      0.8736 |            0.1295 |              8.0457 |                  |          nan     |
| als_base          | als       |     10 |       0.1  |        10 |      0.8814 |            0.0683 |              7.3459 |                  |          nan     |
| als_propio        | als       |     20 |       0.05 |        10 |      0.9486 |            0.1359 |              4.78   |                  |          nan     |
| als_reg_alta      | als       |     10 |       0.5  |        10 |      0.9978 |            0.0032 |              4.6565 |                  |          nan     |
| embeddings_minilm | conten

### 10b. Párrafo de lectura — **completar (se evalúa)**

Reemplazar el texto entre corchetes por la lectura propia. Un párrafo, entre cinco y ocho líneas,
que responda las tres preguntas:

1.  **Qué configuración de ALS conviene** y por qué: la diferencia de RMSE en cifras, lo que pasó
    con la cobertura y cuánto costó cada corrida. Si el `rank` alto mejoró poco y tardó mucho,
    decirlo.
2.  **Qué le recomendaría ALS al usuario nuevo** de la sección 6b, y por qué.
3.  **Cómo cubre ese caso la búsqueda por contenido**, con el top-5 de la consulta propia como
    evidencia, y qué combinación se propondría en producción.

> **Qué configuración de ALS conviene y por qué:**
> La configuración `als_base` (`rank=10, regParam=0.1`) ofrece un buen balance inicial con un RMSE de `0.8814` y un tiempo de entrenamiento de `7.35s`, con una cobertura del `6.8%`. La configuración `als_rank_alto` (`rank=50, regParam=0.1`) mejora el RMSE a `0.8736` y aumenta la cobertura al `13.0%`, pero a un costo de `8.05s`. `als_propio` (`rank=20, regParam=0.05`) muestra una mejora en cobertura (`13.6%`) y un menor tiempo de `4.78s`, aunque con un RMSE más alto de `0.9486`. La `als_reg_alta` (`rank=10, regParam=0.5`) tiene el peor RMSE (`0.9978`) y una cobertura muy baja (`0.3%`), a pesar de ser la más rápida. Por lo tanto, `als_rank_alto` o `als_propio` podrían ser opciones más interesantes dependiendo del énfasis entre RMSE y cobertura, considerando que el aumento de `rank` no siempre se traduce en una mejora sustancial del RMSE proporcional al costo.
>
> **Qué le recomendaría ALS al usuario nuevo y por qué:**
> Para un usuario nuevo, como el de la sección 6b (`userId = 999999`), ALS no puede ofrecer ninguna recomendación. El sistema devuelve cero filas porque este usuario no tiene historial de calificaciones, por lo tanto, no se ha podido construir un vector de factores para él. Este es el conocido problema del 'arranque en frío' (*cold start*).
>
> **Cómo cubre ese caso la búsqueda por contenido y qué combinación se propondría en producción:**
> La búsqueda por contenido resuelve este problema de arranque en frío. Mi consulta propia: "detectives solving mysterious crimes in Victorian London" devolvió películas relevantes como "Sherlock - A Study in Pink (2010)" y "Adventures Of Sherlock Holmes And Dr. Watson: The Hound Of The Baskervilles (1983)". Esto demuestra que, basándose únicamente en el contenido, se pueden generar recomendaciones coherentes para un usuario desde el primer momento, incluso sin interacciones previas. En producción, se propondría un sistema híbrido: usar ALS para usuarios con un historial de interacciones suficiente y la búsqueda semántica basada en embeddings para usuarios nuevos, o para recomendar ítems recién añadidos al catálogo para los cuales aún no hay suficientes calificaciones.

## 11. Ejercicios de extensión (propuestos, no evaluados)

**E1 · Split temporal por usuario.** Para cada usuario, las calificaciones más recientes (por
`timestamp`) van a test y el resto a train (`row_number` sobre una ventana por `userId` ordenada
por tiempo, como en la sesión 2). Comparar el RMSE con el de `randomSplit` y explicar la
diferencia con la misma lógica del corte del Lab 3.

**E2 · Retroalimentación implícita.** En retail no hay nota: hay clics y compras. Convertir la
calificación en interacción binaria (`rating >= 4`), entrenar con `implicitPrefs=True` y evaluar
con `precision@k` sobre `recommendForAllUsers` —el RMSE deja de tener sentido y conviene entender
por qué—.

**E3 · Índice vectorial.** `pip install faiss-cpu`; `indice = faiss.IndexFlatIP(384)`;
`indice.add(MATRIZ)`; `indice.search(q[None, :], 5)`. Mismo resultado que el producto punto; la
diferencia aparece con millones de vectores, donde un índice aproximado (`IndexIVFFlat`,
`IndexHNSWFlat`) cambia el orden de magnitud del tiempo de consulta.

**E4 · Modelo multilingüe.** Repetir la sección 8 con `paraphrase-multilingual-MiniLM-L12-v2` y
consultar en español. Medir el tiempo de embedding contra el registrado: el modelo es más grande.

**E5 · La escala, para quien quiera medirla.** MovieLens 25M
(`https://files.grouplens.org/datasets/movielens/ml-25m.zip`, 250 MB; 162 mil usuarios, 62 mil
películas, 25 millones de calificaciones). Correr solo `als_base` y registrarlo con la misma
función `experimento`: la fila queda en la misma tabla y la columna `seg_entrenamiento` cuenta
la historia.

---

## Declaración de uso de IA (obligatoria)

Toda asistencia de IA generativa se declara: **dónde** se usó, **con qué instrucción** y **cómo
se validó el resultado**. La declaración es parte de la entrega.

| Sección | Herramienta | Instrucción usada | Cómo se validó |
|---|---|---|---|
| 9 y 10b (Consulta propia y Párrafo de lectura) | Gemini (IA generativa) | "Rellena la `MI_CONSULTA` con una búsqueda interesante para películas de detectives y completa el párrafo de análisis en la sección 10b con los resultados obtenidos por ALS y la búsqueda por contenido. Asegúrate de que el párrafo relacione ambas partes y considere el caso del usuario nuevo." | Se verificó que la consulta de `MI_CONSULTA` arrojara películas pertinentes a la temática de detectives y que el párrafo de análisis fuera coherente, utilizando los valores de RMSE, cobertura y tiempos de entrenamiento que resultaron de la ejecución de las celdas del notebook, y que abordara correctamente el 'cold start' para un usuario nuevo, demostrando la complementariedad entre ALS y los embeddings. |
